# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
# TODO: Import the necessary libs
# For example: 
import os
from dotenv import load_dotenv
from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [ ]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [2]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
import chromadb
from lib.tooling import tool
from lib.vector_db import VectorStore

chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")

# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

@tool
def retrieve_game(query: str) -> list[dict]:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.
    """
    vector_store = VectorStore(chroma_collection=collection)
    results = vector_store.query(query_texts=[query], n_results=5)
    return results["documents"]

#### Evaluate Retrieval Tool

In [3]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result
from lib.evaluation import AgentEvaluator, TestCase, EvaluationResult

@tool
def evaluate_retrieval(question: str, retrieved_docs: list[dict]):
    """
    Based on the user's question and on the list of retrieved documents, 
    it will analyze the usability of the documents to respond to that question. 
        args:
        - question: original question from user
        - retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    The result includes:
        - useful: whether the documents are useful to answer the question
        - description: description about the evaluation result
    """
    result = {
        "useful": False,
        "description": ""
    }
    agent_evaluator = AgentEvaluator(api_key=OPENAI_API_KEY, tools=[retrieve_game])
    evaluation = agent_evaluator.evaluate_retrieval(question, retrieved_docs)
    if evaluation.task_completion.task_completed:
        result["useful"] = True
        result["description"] = evaluation.feedback
    else:
        result["useful"] = False
        result["description"] = evaluation.feedback
    return result


#### Game Web Search Tool

In [ ]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry.
@tool
def game_web_search(question: str) -> list[dict]:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - question: a question about game industry.
    """
    results = []
    return results

### Agent

In [ ]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

agentic_rag = Agent(
    model_name="gpt-4o-mini",
    tools=[retrieve_game, game_web_search],
    instructions=(
        "You are an Agentic RAG assistant that can intelligently decide which tools to use "
        "to answer user questions. Reason about the response, change the query and call the tool again if needed "
        "in order to get better results. Always explain your reasoning for tool selection and provide comprehensive answers."
    )
)

In [ ]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?
run_1 = agentic_rag.invoke(
    query="When Pokémon Gold and Silver was released?",
    session_id="pokemon",
)
run_2 = agentic_rag.invoke(
    query="Which one was the first 3D platformer Mario game?",
    session_id="pokemon",
)
run_3 = agentic_rag.invoke(
    query="Was Mortal Kombat X realeased for Playstation 5?",
    session_id="pokemon",
)

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes